# S&P 500 Options: Holdout Backtest

**Chapter 20 - Out-of-sample evaluation**

[`16_holdout_predictions`](16_holdout_predictions.ipynb) refitted the selected
configuration on the history before the holdout window and wrote its predictions over it.
This notebook writes straddles from them, with the concentration, the allocator, the entry
schedule, the hedge rule and the cost assumption the rest of the case study used, and
registers the result.

Nothing is chosen here. The predictions, the allocator, the concentration, the weekly entry
schedule, the delta-hedge threshold and the per-leg charges all arrive fixed from earlier
notebooks, and the only thing this notebook decides is that they are applied unchanged. That
is the whole design: a holdout result is worth something exactly to the extent that no
decision was made after seeing it, and every knob left open here would be a decision.

The comparison to validation is printed but not interpreted. The validation figure is the
maximum of a ranking over the whole sweep and the holdout figure is one measurement over a
single year; what can be said about the gap between them is
[`18_strategy_analysis`](18_strategy_analysis.ipynb)'s subject, with the intervals to say
it.

**Prerequisites:** [`16_holdout_predictions`](16_holdout_predictions.ipynb).

**Scope:** one backtest. No selection, no comparison beyond a printed pair.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# S&P 500 選擇權：保留測試集回測

**第 20 章：樣本外評估**

[`16_holdout_predictions`](16_holdout_predictions.ipynb) 使用保留測試集時間窗之前的歷史資料重新擬合選定組態，並產生該時間窗內的預測。本 Notebook 依案例研究其餘部分使用的持倉集中度、配置方法、進場時程、避險規則及成本假設，將預測轉為跨式組合並登錄結果。

此處不做任何選取。預測、配置方法、持倉集中度、每週進場時程、Delta 避險門檻及雙腿各項費用，均由先前 Notebook 固定；本 Notebook 唯一決定的是原樣套用。這正是整體設計：保留測試集結果的價值，取決於看到它之後沒有再做決策的程度；此處每個未固定的調整鈕，都會成為一項新決策。

此處印出與驗證結果的比較，但不加以解讀。驗證數字是整個掃描排序後的最大值，保留測試集數字則是單一年份的一次量測；如何解讀兩者差距，是 [`18_strategy_analysis`](18_strategy_analysis.ipynb) 的主題，屆時會使用區間支持判斷。

**先備條件：** [`16_holdout_predictions`](16_holdout_predictions.ipynb)。

**範圍：**一次回測。不進行選取；除了印出一對數字外，不做比較。
<!-- ML4T_ZH_END c0000-p000 -->


In [1]:
"""S&P 500 Options: Holdout Backtest."""

import json
import sqlite3
import warnings

import polars as pl

warnings.filterwarnings("ignore")

from case_studies.research import supersedes_for_run
from case_studies.research.holdout import build_holdout_training_spec
from case_studies.sp500_options.research_workflow import (
    open_study,
    run_official_backtest_requests,
    strategy_request_frame,
)
from case_studies.utils.backtest_presets import strategy_view
from case_studies.utils.registry import training_hash_from_spec
from case_studies.utils.strategy_analysis import resolve_solvent_carrier
from utils.paths import get_case_study_dir

In [2]:
CASE_STUDY_ID = "sp500_options"
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
# The generation of the holdout population this run retires. A population is immutable under
# its name, so a re-run whose member has moved has to name the one it replaces; the refusal
# prints the current hash. Empty is correct only for a name this registry has never held.
# It moves whenever `16_holdout_predictions` refits a different configuration, which is what
# an upstream correction does - the notebook carried no declaration at all, so the first such
# correction left it unable to register the backtest it had just run.
SUPERSEDES_HOLDOUT_POPULATION: str = ""

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)
REGISTRY = CASE_DIR / "run_log" / "registry.db"

## 1. The configuration, and the predictions it produced on the holdout

The selected configuration is resolved the same way [`15_costs`](15_costs.ipynb) and
[`16_holdout_predictions`](16_holdout_predictions.ipynb) resolve it, so all three run the
same configuration by construction rather than by a hash copied between them.

Which holdout prediction set belongs to it is derived rather than searched for. Re-deriving the
holdout training specification reproduces the training identity 16 registered - the derivation is
deterministic and the identity covers it - so the prediction set is looked up by that identity
and the selected configuration's checkpoint. A search over holdout prediction sets would have to
guess which one belonged to this configuration, and this registry has held holdout prediction
sets that belonged to no refit at all.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 1. 組態及其在保留測試集產生的預測

選定組態的解析方式，與 [`15_costs`](15_costs.ipynb) 及 [`16_holdout_predictions`](16_holdout_predictions.ipynb) 相同，因此三者依設計使用同一組態，而非在它們之間複製雜湊值。

屬於該組態的保留測試集預測集是推導出來的，不是搜尋出來的。重新推導保留測試集訓練規格，會重現 16 登錄的訓練身分識別值，因為推導具決定性，識別值也涵蓋其內容；因此，預測集依此識別值及選定組態的檢查點查找。若搜尋所有保留測試集預測，就必須猜測哪一組屬於此組態，而這個登錄系統曾存有根本不屬於任何重新擬合結果的保留測試集預測。
<!-- ML4T_ZH_END c0004-p000 -->


In [4]:
carrier = resolve_solvent_carrier(CASE_STUDY_ID)
LABEL = carrier["label"]
validation_prediction_record = study.results.open(carrier["val_prediction_hash"]).registry_record()

holdout_spec = build_holdout_training_spec(
    study,
    study.results.open(carrier["training_hash"]).spec(),
    timeline=(
        pl.read_parquet(study.root / "labels" / f"{LABEL}.parquet")
        .get_column("timestamp")
        .unique()
        .sort()
        .to_list()
    ),
    case_study=CASE_STUDY_ID,
)
holdout_training_hash = training_hash_from_spec(holdout_spec)

with sqlite3.connect(str(REGISTRY)) as conn:
    match = conn.execute(
        """
        SELECT prediction_hash FROM prediction_sets
        WHERE split = 'holdout' AND training_hash = ?
          AND checkpoint_kind IS ? AND checkpoint_value IS ?
        """,
        (
            holdout_training_hash,
            validation_prediction_record["checkpoint_kind"],
            validation_prediction_record["checkpoint_value"],
        ),
    ).fetchone()
if match is None:
    raise RuntimeError(
        f"No holdout prediction set for training {holdout_training_hash}. Run "
        "16_holdout_predictions first; this notebook does not fit."
    )
HOLDOUT_PREDICTION_HASH = match[0]

print(f"Selected configuration: {carrier['val_backtest_hash']}  {carrier['config_name']} ({LABEL})")
print(f"Holdout training:   {holdout_training_hash}")
print(f"Holdout prediction: {HOLDOUT_PREDICTION_HASH}")

Selected configuration: 52779803a485  default_mae (ret_to_expiry)
Holdout training:   ac02c6e665be
Holdout prediction: 2ddee8bd10d0


## 2. The request, and what may run on the window

The strategy is the selected configuration's own, read off its registered specification and
re-pointed at the holdout prediction set. `signal` carries the entry schedule, the
liquid-universe filter and the concentration; `allocation` carries the sizing. Neither is retyped
here, so a rebuilt sweep cannot leave this notebook running last quarter's strategy.

**The window carries one backtest.** 16's guard is on the model - the training identity and
the checkpoint - and it cannot see this one: a changed allocator, concentration or cost
level produces the same holdout predictions and a different result from them. Two things
enforce it, and they catch different mistakes. The check below refuses a holdout-stage
backtest that belongs to some other prediction set, which is a second *configuration*
measured on the window. The official population this run publishes refuses a second
*strategy* over the same predictions: the name already exists with a different member list,
and `create` will not overwrite it without being told which generation it replaces, which
this notebook does not say.

Both refuse rather than offering a replacement switch. Deleting a result that has been seen
does not unsee it, and the selection bias it introduces is not removed by removing the rows.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 2. 請求，以及哪些內容可在此時間窗執行

策略是選定組態原本的策略，從已登錄規格讀取，再改指向保留測試集預測集。`signal` 帶入進場時程、流動性標的範圍篩選及持倉集中度；`allocation` 帶入部位規模。兩者都不在此重新手動輸入，因此即使掃描重新建立，本 Notebook 也不會繼續使用上一季的策略。

**此時間窗只容納一次回測。** 16 的防護針對模型，也就是訓練身分識別值與檢查點，無法看見此處的問題：改變配置方法、持倉集中度或成本水準，會使用同一組保留測試集預測，卻產生不同結果。以下兩項機制共同執行限制，分別捕捉不同錯誤。下方檢查拒絕屬於其他預測集的保留測試集階段回測，避免在同一時間窗量測第二種*組態*。本次執行發布的正式母體，則會拒絕針對同一組預測的第二種*策略*：該名稱已對應不同成員清單，而 `create` 未獲明確指定要取代哪一代時，不會覆寫；本 Notebook 沒有指定。

兩項機制都直接拒絕，不提供替換開關。刪除看過的結果無法讓人忘記結果，刪除資料列也無法消除因此產生的選取偏誤。
<!-- ML4T_ZH_END c0006-p000 -->


In [5]:
with sqlite3.connect(str(REGISTRY)) as conn:
    foreign_holdout_backtests = [
        backtest_hash
        for (backtest_hash,) in conn.execute(
            """
            SELECT b.backtest_hash FROM backtest_runs b
            JOIN prediction_sets p ON p.prediction_hash = b.prediction_hash
            WHERE p.split = 'holdout' AND b.prediction_hash != ?
            ORDER BY b.backtest_hash
            """,
            (HOLDOUT_PREDICTION_HASH,),
        ).fetchall()
    ]
if foreign_holdout_backtests:
    raise RuntimeError(
        "the holdout window already carries a backtest of a different configuration: "
        + ", ".join(foreign_holdout_backtests)
        + ". This run has not started. Same rule as 16_holdout_predictions: discarding the "
        "earlier result would not undo having observed it, so there is no switch here. Leave "
        "the selection where it was, or retire the earlier evaluation through the registry's "
        "lifecycle."
    )

carrier_strategy = strategy_view(json.loads(carrier["spec_json"]))
SIGNAL = carrier_strategy["signal"]
ALLOCATION = carrier_strategy.get("allocation")
HOLDOUT_POPULATION = f"{CASE_STUDY_ID}-holdout-{LABEL}"
print(f"Signal:     {SIGNAL}")
print(f"Allocation: {ALLOCATION}")

Signal:     {'direction': 'long_only', 'long_short': False, 'method': 'equal_weight_top_k', 'schedule_contract': 'last_available_session_per_iso_week_v1', 'top_k': 5, 'universe_filter': 'liquid'}
Allocation: {'method': 'score_weighted'}


## 3. The backtest

The run goes through `run_official_backtest_requests`, the same call the validation sweeps
use, with `split="holdout"` naming the price window. It is a parameter rather than a second
code path on purpose: the entry schedule, the delta hedge and the settlement at intrinsic
value are identical in both windows and only the prices differ, so a holdout-specific copy of
that function would be a copy that can drift from the one every validation number came from.

What that call does here is what it does everywhere in this case study. It resolves the short
straddles the predictions imply, replays that resolution in a fresh interpreter and refuses
to continue if the two disagree, publishes the decision artifact, computes the backtest
identity before executing, and registers the run under it.

The run registers under `stage='holdout'`, which the registry derives from the prediction
set's split rather than from anything asserted here. That is checked below rather than
trusted: if the inference ever stopped taking precedence, the whole out-of-sample result
would land in `allocation` and 18 would find no holdout at all, which is a failure two
notebooks away from its cause.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 3. 回測

本次執行使用 `run_official_backtest_requests`，與驗證掃描相同的呼叫，並以 `split="holdout"` 指定價格時間窗。刻意將其設為參數，而非第二條程式路徑：兩個時間窗的進場時程、Delta 避險及按內在價值結算均相同，只有價格不同；若複製一份專供保留測試集使用的函式，就可能逐漸偏離產生所有驗證數字的原函式。

此呼叫在這裡所做的，與本案例其他地方相同。它解析預測所對應的空頭跨式組合，在新的直譯器中重播解析過程；若兩者不一致，就拒絕繼續；接著發布決策資料產物，在執行前計算回測身分識別值，並依該值登錄執行結果。

本次執行登錄在 `stage='holdout'` 之下；登錄系統根據預測集的資料切分推導此值，而非相信此處的宣稱。下方仍會檢查：若推導的優先順序失效，整個樣本外結果就會落在 `allocation`，而 18 會找不到保留測試集；錯誤要到兩份 Notebook 之後才會顯現。
<!-- ML4T_ZH_END c0008-p000 -->


In [6]:
requests = strategy_request_frame(
    [
        {
            "request_name": f"holdout-{carrier['config_name']}",
            "prediction_hash": HOLDOUT_PREDICTION_HASH,
            "label": LABEL,
            "signal": SIGNAL,
            "allocation": ALLOCATION,
            "chapter": "ch20",
        }
    ]
)
execution = run_official_backtest_requests(
    study,
    requests,
    population_name=HOLDOUT_POPULATION,
    split="holdout",
    supersedes=supersedes_for_run(
        study,
        population_name=HOLDOUT_POPULATION,
        declared=SUPERSEDES_HOLDOUT_POPULATION or None,
        execution_tier=EXECUTION_TIER,
    ),
)
result = execution.results[0]
print(f"Holdout backtest: {result.hash}")
execution.catalog_rows

  SKIP backtest (complete (hash=6e9f58e1fd42)) — reusing cached result
Holdout backtest: 6e9f58e1fd42


request_name,label,prediction_hash,decision_hash,backtest_hash,complete
str,str,str,str,str,bool
"""holdout-default_mae""","""ret_to_expiry""","""2ddee8bd10d0""","""8b267f21da72""","""6e9f58e1fd42""",true


The registered run is read back rather than described from the request. The strategy view of what
was registered has to equal the selected configuration's, or this is a different strategy wearing
the configuration's name - the specification is rebuilt from `signal` and `allocation` here, and
the rebalance cadence and cost levels it fills in around them come from `setup.yaml`, so the
comparison is what establishes that they filled in the same way they did on validation.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
回讀已登錄執行結果，而非根據請求描述它。登錄策略的視圖必須與選定組態相同，否則就是不同策略冒用該組態名稱。此處由 `signal` 與 `allocation` 重新建立規格，而補足再平衡頻率及成本水準的設定來自 `setup.yaml`；這項比較才能證明補足方式與驗證期間一致。
<!-- ML4T_ZH_END c0010-p000 -->


In [7]:
with sqlite3.connect(str(REGISTRY)) as conn:
    registered_stage, registered_spec_json = conn.execute(
        "SELECT stage, spec_json FROM backtest_runs WHERE backtest_hash = ?", (result.hash,)
    ).fetchone()
if registered_stage != "holdout":
    raise RuntimeError(
        f"the holdout backtest registered under stage={registered_stage!r} rather than "
        "'holdout'; the split-based inference in registry.store._infer_stage did not take "
        "precedence over this configuration's allocation block"
    )
registered_strategy = strategy_view(json.loads(registered_spec_json))
if registered_strategy != carrier_strategy:
    raise RuntimeError(
        "the holdout backtest did not register the selected configuration's strategy. Configuration: "
        f"{carrier_strategy}. Registered: {registered_strategy}."
    )
print(f"stage={registered_stage}, strategy matches the selected configuration")
print(f"Official population {HOLDOUT_POPULATION}: {execution.population.hash}")

stage=holdout, strategy matches the selected configuration
Official population sp500_options-holdout-ret_to_expiry: f557d0d1b51a


## 4. What it came out at

The two numbers below are one strategy measured on two disjoint periods, and the gap between
them is not an estimate of decay. The validation figure is the maximum of a ranking over the
whole sweep, so it carries the selection; the holdout figure is one measurement over a single
year, so it carries that window's sampling error. Both push the pair apart on their own,
before any real change in the strategy's edge - and this configuration's validation Sharpe is
already negative, so what the holdout can confirm or disturb is a negative result rather than
a positive one.

[`18_strategy_analysis`](18_strategy_analysis.ipynb) is where they are given intervals and a
paired comparison.

<!-- ML4T_ZH_BEGIN c0012-p000 -->
## 4. 得到的數字

下方兩個數字，是同一策略在兩個互不重疊期間的量測；其差距不是效果衰退的估計值。驗證數字是整個掃描排序後的最大值，因此帶有選取影響；保留測試集數字是單一年份的一次量測，因此帶有該時間窗的抽樣誤差。即使策略的真實優勢完全未變，兩者也足以讓這對數字分開。而且此組態的驗證夏普比率本來就是負的，因此保留測試集能確認或動搖的是負面結果，而非正面結果。

[`18_strategy_analysis`](18_strategy_analysis.ipynb) 會為兩者附上區間並進行配對比較。
<!-- ML4T_ZH_END c0012-p000 -->


In [8]:
METRIC_COLUMNS = ("sharpe", "cagr", "max_drawdown", "win_rate", "n_periods")
with sqlite3.connect(str(REGISTRY)) as conn:
    columns = ", ".join(METRIC_COLUMNS)
    carrier_metrics, holdout_metrics = (
        dict(
            zip(
                METRIC_COLUMNS,
                conn.execute(
                    f"SELECT {columns} FROM backtest_metrics WHERE backtest_hash = ?",
                    (backtest_hash,),
                ).fetchone(),
                strict=True,
            )
        )
        for backtest_hash in (carrier["val_backtest_hash"], result.hash)
    )

# The selected configuration's own registered Sharpe, not the resolver's. `resolve_solvent_carrier`
# reports the common-support figure, which re-ranks candidates on the timestamps every one of them
# covers; that is the right number for choosing between candidates and the wrong one to set beside a
# holdout measured over its own full window. Both are printed, so neither has to be inferred from
# the other.
#
# `num_trades` is deliberately absent. The HTM cohort engine settles each straddle at expiration
# rather than rebalancing a book, so it registers no trade count for either run, and printing a
# zero would read as "it stopped trading" rather than "this engine does not count trades".
for name, metrics in (("Validation", carrier_metrics), ("Holdout   ", holdout_metrics)):
    print(
        f"{name} over {int(metrics['n_periods']):,} sessions: Sharpe "
        f"{metrics['sharpe']:.3f}, CAGR {metrics['cagr']:.1%}, "
        f"max drawdown {metrics['max_drawdown']:.2%}, win rate {metrics['win_rate']:.0%}"
    )
print(f"  the validation run re-ranked on common support: {carrier['val_sharpe']:.3f}")

Validation over 473 sessions: Sharpe -0.102, CAGR -21.9%, max drawdown -77.60%, win rate 61%
Holdout    over 247 sessions: Sharpe 0.588, CAGR 17.2%, max drawdown -35.34%, win rate 58%
  the validation run re-ranked on common support: -0.182


## What this notebook establishes, and what it does not

It establishes a return series for the selected configuration over a period no choice in this
case study was made on. That is the only thing a holdout can give, and it is worth less than
it looks: one year of weekly straddle cohorts is a small number of independent observations,
which is too few to separate a strategy that decayed from one that had an ordinary year.

It does not establish that this configuration was the right one to carry here. The selection
that brought it was made on validation, over a pool large enough that its maximum is
optimistic by construction, and this notebook inherits that pool without correcting for it.
The deflation is [`18_strategy_analysis`](18_strategy_analysis.ipynb)'s.

Re-running this notebook against the same configuration is free and idempotent - the backtest
identity is unchanged and the registered run is served back. A different configuration is refused,
for the reason 16 gives.

**Next:** [`18_strategy_analysis`](18_strategy_analysis.ipynb).

<!-- ML4T_ZH_BEGIN c0014-p000 -->
## 本 Notebook 證明了什麼，以及尚未證明什麼

它建立了選定組態在本案例未用於任何選取的期間內的報酬序列。這是保留測試集唯一能提供的內容，而且價值比表面看來更有限：一年內每週建立的跨式組合批次，只提供少量獨立觀測，不足以區分策略優勢衰退與經歷普通年份的策略。

它不能證明帶到此處的組態選得正確。該組態是從驗證結果選出，候選池大到其最大值依設計就偏樂觀；本 Notebook 沿用該候選池，沒有修正。[`18_strategy_analysis`](18_strategy_analysis.ipynb) 會處理這項校正。

對同一組態重新執行本 Notebook 不會重做回測，且具冪等性：回測身分識別值不變，登錄系統會直接提供既有執行結果。依 16 所述理由，系統會拒絕不同組態。

**下一步：** [`18_strategy_analysis`](18_strategy_analysis.ipynb)。
<!-- ML4T_ZH_END c0014-p000 -->
